<h2> 3.6 Featurizing text data with tfidf weighted word-vectors </h2>

In [ ]:
!pip install spacy
import pandas as pd
import matplotlib.pyplot as plt
import re
import time
import warnings
import numpy as np
from nltk.corpus import stopwords
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
warnings.filterwarnings("ignore")
import sys
import os 
import pandas as pd
import numpy as np
from tqdm import tqdm

# exctract word2vec vectors
# https://github.com/explosion/spaCy/issues/1721
# http://landinghub.visualstudio.com/visual-cpp-build-tools
import spacy

In [ ]:
# avoid decoding problems
df = pd.read_csv("train.csv")
new_df = df.sample(n=90000, random_state=1)
df['question1'] = df['question1'].apply(lambda x: str(x))
df['question2'] = df['question2'].apply(lambda x: str(x))

In [ ]:
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# merge texts
questions = list(df['question1']) + list(df['question2'])

tfidf = TfidfVectorizer(lowercase=False, )
tfidf.fit_transform(questions)

# dict key:word and value:tf-idf score
word2tfidf = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

- After we find TF-IDF scores, we convert each question to a weighted average of word2vec vectors by these scores.
- here we use a pre-trained GLOVE model which comes free with "Spacy".  https://spacy.io/usage/vectors-similarity
- It is trained on Wikipedia and therefore, it is stronger in terms of word semantics. 

In [ ]:
!pip install --upgrade numpy h5py spacy


In [ ]:
!python -m spacy download en_core_web_sm


In [ ]:
# en_vectors_web_lg, which includes over 1 million unique vectors.
nlp = spacy.load('en_core_web_sm')

vecs1 = []
# https://github.com/noamraph/tqdm
# tqdm is used to print the progress bar
for qu1 in tqdm(list(df['question1'])):
    doc1 = nlp(qu1) 
    # 384 is the number of dimensions of vectors 
    mean_vec1 = np.zeros([len(doc1), len(doc1[0].vector)])
    for word1 in doc1:
        # word2vec
        vec1 = word1.vector
        # fetch df score
        try:
            idf = word2tfidf[str(word1)]
        except:
            idf = 0
        # compute final vec
        mean_vec1 += vec1 * idf
    mean_vec1 = mean_vec1.mean(axis=0)
    vecs1.append(mean_vec1)
df['q1_feats_m'] = list(vecs1)


In [ ]:
print(len(doc[0].vector))

In [ ]:
vecs2 = []
for qu2 in tqdm(list(df['question2'])):
    doc2 = nlp(qu2) 
    mean_vec2 = np.zeros([len(doc1), len(doc2[0].vector)])
    for word2 in doc2:
        # word2vec
        vec2 = word2.vector
        # fetch df score
        try:
            idf = word2tfidf[str(word2)]
        except:
            #print word
            idf = 0
        # compute final vec
        mean_vec2 += vec2 * idf
    mean_vec2 = mean_vec2.mean(axis=0)
    vecs2.append(mean_vec2)
df['q2_feats_m'] = list(vecs2)

In [ ]:
#preprocessed_data(Basic).csv (Simple Preprocessing Feartures)
#preprocessed_data.csv (NLP Features)
if os.path.isfile('preprocessed_data.csv'):
    dfnlp = pd.read_csv("preprocessed_data.csv",encoding='latin-1')
else:
    print("download preprocessed_data.csv from drive or run previous notebook")

if os.path.isfile('preprocessed_data(Basic).csv'):
    dfppro = pd.read_csv("preprocessed_data(Basic).csv",encoding='latin-1')
else:
    print("download preprocessed_data(Basic).csv from drive or run previous notebook")

In [ ]:
dfnlp.head()

In [ ]:
df1 = dfnlp.drop(['qid1','qid2','question1','question2'],axis=1)
df2 = dfppro.drop(['qid1','qid2','question1','question2','is_duplicate'],axis=1)
df3 = df.drop(['qid1','qid2','question1','question2','is_duplicate'],axis=1)


In [ ]:
df3_q1 = pd.DataFrame(df3.q1_feats_m.values.tolist(), index= df3.index)
df3_q2 = pd.DataFrame(df3.q2_feats_m.values.tolist(), index= df3.index)

In [ ]:
# dataframe of nlp features
df1.head()

In [ ]:
# data before preprocessing 
df2.head()

In [ ]:
# Questions 1 tfidf weighted word2vec
df3_q1.head()

In [ ]:
# Questions 2 tfidf weighted word2vec
df3_q2.head()

In [ ]:
print("Number of features in nlp dataframe :", df1.shape[1])
print("Number of features in preprocessed dataframe :", df2.shape[1])
print("Number of features in question1 w2v  dataframe :", df3_q1.shape[1])
print("Number of features in question2 w2v  dataframe :", df3_q2.shape[1])
print("Number of features in final dataframe  :", df1.shape[1]+df2.shape[1]+df3_q1.shape[1]+df3_q2.shape[1])

In [ ]:

if not os.path.isfile('final_features.csv'):

    # 1. Ensure all dfs only keep ids that exist in df1
    valid_ids = df1['id'].unique()

    df2 = df2[df2['id'].isin(valid_ids)].drop_duplicates(subset=['id'])
    df3_q1 = df3_q1[df3_q1['id'].isin(valid_ids)].drop_duplicates(subset=['id'])
    df3_q2 = df3_q2[df3_q2['id'].isin(valid_ids)].drop_duplicates(subset=['id'])

    # 2. Debug shapes after cleaning
    print("Shapes after aligning IDs:")
    print("df1:", df1.shape)
    print("df2:", df2.shape)
    print("df3_q1:", df3_q1.shape)
    print("df3_q2:", df3_q2.shape)

    # 3. Merge df1 + df2 (basic + NLP features etc.)
    df1 = df1.merge(df2, on='id', how='left', suffixes=('_main', '_df2'))

    # 4. Merge df3_q1 + df3_q2 (question features)
    df_q = df3_q1.merge(df3_q2, on='id', how='left', suffixes=('_q1', '_q2'))

    # 5. Merge everything
    result = df1.merge(df_q, on='id', how='left')

    # 6. Print final shape & save
    print("? Final merged shape:", result.shape)
    result.to_csv('final_features.csv', index=False)
    print("? final_features.csv saved successfully!")

else:
    print("final_features.csv already exists.")


In [ ]:

print(result.shape)

In [ ]:
result.head()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix


In [ ]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
# Initialize Random Forest
rf = RandomForestClassifier(
    n_estimators=200,   # number of trees
    max_depth=10,       # max depth of each tree
    random_state=42,
    n_jobs=-1
)

# Train
rf.fit(X_train, y_train)

# Predict
y_pred_rf = rf.predict(X_test)

# Metrics
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Random Forest F1 Score:", f1_score(y_test, y_pred_rf))
print("Random Forest ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))


In [ ]:
# Initialize XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42,
    n_jobs=-1
)

# Train
xgb.fit(X_train, y_train)

# Predict
y_pred_xgb = xgb.predict(X_test)

# Metrics
print("XGBoost Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("XGBoost F1 Score:", f1_score(y_test, y_pred_xgb))
print("XGBoost ROC-AUC:", roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
